In [ ]:
from libraries import *
from parameters import *
import anndata as ad

In [ ]:
%load_ext rpy2.ipython

In [ ]:
os.getcwd()
os.chdir(projectDir)

In [ ]:
adataLPSMinus = sc.read_10x_h5('./data/data_unperturbed/LPSMinus_v3_raw_feature_bc_matrix.h5').copy()
adataLPSPlus = sc.read_10x_h5('./data/data_unperturbed/LPSPlus_v3_raw_feature_bc_matrix.h5').copy()
adataLPSMinus.var_names_make_unique()
adataLPSPlus.var_names_make_unique()

In [ ]:
sc.pp.filter_cells(adataLPSMinus, min_genes=800)
sc.pp.filter_genes(adataLPSMinus, min_cells=10)
sc.pp.filter_cells(adataLPSPlus, min_genes=800)
sc.pp.filter_genes(adataLPSPlus, min_cells=10)

In [ ]:
adataLPSMinus.var['mt'] = adataLPSMinus.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adataLPSMinus, 
                           qc_vars=['mt'], 
                           percent_top=None, 
                           log1p=False, 
                           inplace=True)
adataLPSPlus.var['mt'] = adataLPSPlus.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adataLPSPlus, 
                           qc_vars=['mt'], 
                           percent_top=None, 
                           log1p=False, 
                           inplace=True)

In [ ]:
sc.pl.violin(adataLPSMinus, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

sc.pl.violin(adataLPSPlus, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [ ]:
adataLPSPlus = adataLPSPlus[adataLPSPlus.obs.n_genes_by_counts < 2500, :]
adataLPSPlus = adataLPSPlus[adataLPSPlus.obs.pct_counts_mt < 10, :]
adataLPSMinus = adataLPSMinus[adataLPSMinus.obs.n_genes_by_counts < 2500, :]
adataLPSMinus = adataLPSMinus[adataLPSMinus.obs.pct_counts_mt < 10, :]

In [ ]:
adataLPSPlus.obs["type"] = "LPSPlus"
adataLPSMinus.obs["type"] = "LPSMinus"

In [ ]:
adataAll = ad.concat([adataLPSMinus, adataLPSPlus], join="outer")
#adataAll.var_names_make_unique()
adataAll.obs_names_make_unique()

In [ ]:
sc.pp.normalize_total(adataAll, target_sum=1e4)
sc.pp.log1p(adataAll)
adataAll.raw = adataAll


In [ ]:
sc.pp.highly_variable_genes(adataAll, n_top_genes = 2000)
sc.pp.regress_out(adataAll, ['total_counts', 'pct_counts_mt'])
sc.pp.scale(adataAll, max_value=10)
sc.tl.pca(adataAll, svd_solver='arpack')
sc.pp.neighbors(adataAll, n_neighbors=10, n_pcs=50)


In [ ]:
sc.tl.umap(adataAll)

In [ ]:
sc.tl.leiden(adataAll, resolution=0.5)

In [ ]:
sc.tl.diffmap(adataAll)

In [ ]:
sc.tl.rank_genes_groups(adataAll, 
                        groupby="leiden", 
                        n_genes=2000, 
                        method="t-test_overestim_var")

In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adataAll, color='leiden', 
           legend_loc='on data', 
           legend_fontoutline=3, 
           legend_fontsize=14, 
           legend_fontweight='normal', 
           title='Clusters', 
           ax=ax, 
           show=False, 
           size=3);

In [ ]:
sc.pl.umap(adataAll, color='type', 
           color_map="coolwarm", 
           legend_loc='on data', 
           legend_fontoutline=3, 
           legend_fontsize=14, 
           legend_fontweight='normal', 
           size=3);

In [ ]:
sc.tl.dendrogram(adataAll, groupby='leiden')

In [ ]:
dcGenes = pd.read_csv('/home/beraslan/jovian-work/analysisSingle/PositiveControls/DC_cellstate_genes.csv')
dc1Genes = dcGenes["DC1 genes"].unique()

In [ ]:
sc.tl.score_genes(adata=adataAll, gene_list=dc1Genes, score_name="DC1")
sc.pl.umap(adataAll, color="DC1", 
           size=4, 
           color_map="coolwarm", 
           vmax=0.1, 
           vmin=-0.1)


In [ ]:
sc.pl.violin(adataAll, "DC1", groupby='leiden')

In [ ]:
dc2Genes = dcGenes["DC2 genes"].unique()
sc.tl.score_genes(adata=adataAll, gene_list=dc2Genes, score_name="DC2")
sc.pl.umap(adataAll, 
           color="DC2", 
           size=4, 
           color_map="coolwarm", 
           vmin=-0.1)


In [ ]:
sc.pl.violin(adataAll, "DC2", groupby='leiden')

In [ ]:
mregGenes = dcGenes["mregDC genes"].unique()
sc.tl.score_genes(adata=adataAll, 
                  gene_list=mregGenes, 
                  score_name="mreg")
sc.pl.umap(adataAll, color="mreg", size=3, color_map="coolwarm")

In [ ]:
sc.pl.violin(adataAll, "mreg", groupby='leiden')

In [ ]:
macGenes = dcGenes["Macrophage genes"].unique()
sc.tl.score_genes(adata=adataAll, gene_list=macGenes, score_name="Mac")
sc.pl.umap(adataAll, 
           color="Mac", 
           size=1, 
           color_map="coolwarm", 
           vmax=0.2, 
           vmin=-0.2)


In [ ]:
sc.pl.violin(adataAll, "Mac", groupby='leiden')

In [ ]:
allDCgenes = np.concatenate((dc1Genes, dc2Genes, mregGenes))
sc.tl.score_genes(adata=adataAll, gene_list=allDCgenes, score_name="DCSig")
sc.pl.umap(adataAll, color="DCSig", 
           size=4, 
           color_map="coolwarm", 
           vmin=-0.2, 
           vmax=0.3)

In [ ]:
adataAll.obs["DCSig_zscore"] = scipy.stats.zscore(adataAll.obs["DCSig"])
adataAll.obs["Mac_zscore"] = scipy.stats.zscore(adataAll.obs["Mac"])

In [ ]:
adataAll.obs["MACoverDC"] = adataAll.obs["Mac_zscore"] - adataAll.obs["DCSig_zscore"]

In [ ]:
sc.pl.umap(adataAll, color="MACoverDC", size=4, color_map="PiYG", vmin=-3, vmax=3)


In [ ]:
gene_list_url = 'https://raw.githubusercontent.com/theislab/scanpy_usage/master/180209_cell_cycle/data/regev_lab_cell_cycle_genes.txt'

cell_cycle_genes = [str(x.strip(), 'utf-8').capitalize() for x in urlopen(gene_list_url)] # capitalize = shame


s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]


sc.tl.score_genes_cell_cycle(adataAll, s_genes=s_genes, g2m_genes=g2m_genes)

In [ ]:
sc.pl.umap(adataAll, color='phase', palette = "Paired" )

In [ ]:
leidenMarkersOfPerturbedCells = pd.read_csv("/home/beraslan/jovian-work/analysisSingle/Leiden_top100Markers.csv")

In [ ]:
leidenMarkersOfPerturbedCells = leidenMarkersOfPerturbedCells.groupby('group').head(50)

In [ ]:
for i in leidenMarkersOfPerturbedCells.group.unique():
     myGeneList = leidenMarkersOfPerturbedCells.loc[leidenMarkersOfPerturbedCells.group == i,'names']
     myGeneList = [x.replace(".","-") for x in myGeneList]
     sc.tl.score_genes(adata=adataAll, gene_list=myGeneList, score_name="LeidenPert "+str(i))
     sc.pl.umap(adataAll, color="LeidenPert "+str(i), size=4, color_map="seismic")
     sc.pl.violin(adataAll, "LeidenPert "+str(i), groupby='leiden')

In [ ]:
markerGenes = pd.DataFrame(adataAll.uns['rank_genes_groups']['names'])
markerGenes = markerGenes.iloc[0:10,:]
markerGenes = np.unique(markerGenes.values.flatten())

In [ ]:
sc.tl.rank_genes_groups(adataAll, 'leiden', 
                        method='t-test_overestim_var', 
                        key_added = "t-test_ov")


In [ ]:
sc.pl.rank_genes_groups_heatmap(adataAll, n_genes=15,
                                key="t-test_ov", 
                                groupby="leiden", 
                                show_gene_labels=True, 
                                vmax=5)


In [ ]:
sc.pl.rank_genes_groups_dotplot(adataAll, n_genes=10, 
                                key="t-test_ov", 
                                groupby="leiden")


In [ ]:
sc.pl.rank_genes_groups_matrixplot(adataAll, n_genes=10, 
                                   key="t-test_ov", 
                                   groupby="leiden")


In [ ]:
sc.tl.rank_genes_groups(adataAll, 'type', method='t-test_overestim_var')


In [ ]:
DEgenes = sc.get.rank_genes_groups_df(adataAll, group='LPSPlus')

In [ ]:
DEgenes = DEgenes.loc[DEgenes.pvals_adj < 0.05,:]

In [ ]:
DEgenes.loc[np.abs(DEgenes.logfoldchanges) > 0.5,:]

In [ ]:
DEgenes = DEgenes.loc[np.abs(DEgenes.logfoldchanges) > 0.5,:]

In [ ]:
DEgenes[["TestCondition"]] = "LPSPlusOverMinus"

In [ ]:
DEgenes.to_csv("LPSPlusOverMinusDEGenes.csv", index=False)

In [ ]:
sns.boxplot(x=DEgenes["logfoldchanges"])

In [ ]:
pd.DataFrame({"GeneNames":adataAll.uns['rank_genes_groups']['names'],
              "logFC":adataAll.uns['rank_genes_groups']['logfoldchanges'],
              "pValAdj":adataAll.uns['rank_genes_groups']['pvals_adj']})


In [ ]:
result = adataAll.uns['rank_genes_groups']
groups = result['names'].dtype.names
pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'pvals']})
